In [1]:
# ============================================
# MODULE 4: SOURCE VERIFICATION & CREDIBILITY CHECKER
# ============================================

import pandas as pd
import re
from urllib.parse import urlparse
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================
# STEP 1: LOAD SOURCES DATABASE
# ============================================

print("="*60)
print("LOADING SOURCES DATABASE")
print("="*60)

# Create sources database with credibility scores
sources = pd.DataFrame({
    "source": [
        "reuters.com",
        "bbc.com",
        "dawn.com",
        "geo.tv",
        "arynews.tv",
        "cnn.com",
        "aljazeera.com",
        "nytimes.com",
        "washingtonpost.com",
        "guardian.com",
        "apnews.com",
        "theguardian.com",
        "wsj.com",
        "ft.com",
        "bloomberg.com",
        "abcnews.com",
        "nbcnews.com",
        "cbsnews.com",
        "foxnews.com",
        "npr.org",
        "theatlantic.com",
        "economist.com",
        "newyorker.com",
        "time.com",
        "newsweek.com",
        "hindustantimes.com",
        "timesofindia.com",
        "scmp.com",
        "japantimes.co.jp",
        "thehindu.com",
        "tribune.com.pk",
        "pakistantoday.com.pk",
        "thefridaytimes.com"
    ],
    "score": [
        98,  # reuters.com
        95,  # bbc.com
        90,  # dawn.com
        85,  # geo.tv
        85,  # arynews.tv
        95,  # cnn.com
        92,  # aljazeera.com
        95,  # nytimes.com
        93,  # washingtonpost.com
        90,  # guardian.com
        96,  # apnews.com
        90,  # theguardian.com
        92,  # wsj.com
        91,  # ft.com
        90,  # bloomberg.com
        88,  # abcnews.com
        88,  # nbcnews.com
        88,  # cbsnews.com
        80,  # foxnews.com
        85,  # npr.org
        85,  # theatlantic.com
        90,  # economist.com
        85,  # newyorker.com
        85,  # time.com
        82,  # newsweek.com
        85,  # hindustantimes.com
        85,  # timesofindia.com
        85,  # scmp.com
        85,  # japantimes.co.jp
        85,  # thehindu.com
        80,  # tribune.com.pk
        75,  # pakistantoday.com.pk
        80   # thefridaytimes.com
    ]
})

print(f"✓ Loaded {len(sources)} trusted sources")
print("\nSources Database (first 5):")
print(sources.head())

LOADING SOURCES DATABASE
✓ Loaded 33 trusted sources

Sources Database (first 5):
        source  score
0  reuters.com     98
1      bbc.com     95
2     dawn.com     90
3       geo.tv     85
4   arynews.tv     85


In [3]:
# ============================================
# STEP 2: LOAD AND COMBINE ALL NEWS DATASETS
# ============================================

print("\n" + "="*60)
print("LOADING AND COMBINING DATASETS")
print("="*60)

datasets = []

# 1. Load fake.csv
try:
    fake = pd.read_csv("../datasets/fake.csv")
    fake['label'] = 0
    fake['source'] = 'fake'
    datasets.append(fake)
    print(f"✓ Loaded fake.csv: {fake.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load fake.csv: {e}")

# 2. Load True.csv
try:
    true = pd.read_csv("../datasets/True.csv")
    true['label'] = 1
    true['source'] = 'true'
    datasets.append(true)
    print(f"✓ Loaded True.csv: {true.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load True.csv: {e}")

# 3. Load AG_Train.csv
try:
    ag_train = pd.read_csv("../datasets/AG_Train.csv")
    
    # Map labels: 3,4 = True (1), others = Fake (0)
    if 'Class Index' in ag_train.columns:
        ag_train['label'] = ag_train['Class Index'].apply(lambda x: 1 if x in [3, 4] else 0)
        # Map categories
        category_map = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
        ag_train['category'] = ag_train['Class Index'].map(category_map)
    elif 'class' in ag_train.columns:
        ag_train['label'] = ag_train['class'].apply(lambda x: 1 if x in [3, 4] else 0)
        category_map = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
        ag_train['category'] = ag_train['class'].map(category_map)
    else:
        first_col = ag_train.columns[0]
        ag_train['label'] = ag_train[first_col].apply(lambda x: 1 if x in [3, 4] else 0)
        category_map = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
        ag_train['category'] = ag_train[first_col].map(category_map)
    
    # Combine Title and Description
    if 'Title' in ag_train.columns and 'Description' in ag_train.columns:
        ag_train['text'] = ag_train['Title'].fillna('') + ' ' + ag_train['Description'].fillna('')
    elif 'title' in ag_train.columns and 'description' in ag_train.columns:
        ag_train['text'] = ag_train['title'].fillna('') + ' ' + ag_train['description'].fillna('')
    elif len(ag_train.columns) >= 3:
        col_names = ag_train.columns.tolist()
        ag_train['text'] = ag_train[col_names[1]].fillna('') + ' ' + ag_train[col_names[2]].fillna('')
    else:
        text_col = ag_train.columns[1] if len(ag_train.columns) > 1 else ag_train.columns[0]
        ag_train['text'] = ag_train[text_col]
    
    ag_train['source'] = 'AG_Train'
    datasets.append(ag_train)
    print(f"✓ Loaded AG_Train.csv: {ag_train.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load AG_Train.csv: {e}")

# 4. Load AG_Test.csv
try:
    ag_test = pd.read_csv("../datasets/AG_Test.csv")
    
    if 'Class Index' in ag_test.columns:
        ag_test['label'] = ag_test['Class Index'].apply(lambda x: 1 if x in [3, 4] else 0)
        category_map = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
        ag_test['category'] = ag_test['Class Index'].map(category_map)
    elif 'class' in ag_test.columns:
        ag_test['label'] = ag_test['class'].apply(lambda x: 1 if x in [3, 4] else 0)
        category_map = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
        ag_test['category'] = ag_test['class'].map(category_map)
    else:
        first_col = ag_test.columns[0]
        ag_test['label'] = ag_test[first_col].apply(lambda x: 1 if x in [3, 4] else 0)
        category_map = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
        ag_test['category'] = ag_test[first_col].map(category_map)
    
    if 'Title' in ag_test.columns and 'Description' in ag_test.columns:
        ag_test['text'] = ag_test['Title'].fillna('') + ' ' + ag_test['Description'].fillna('')
    elif 'title' in ag_test.columns and 'description' in ag_test.columns:
        ag_test['text'] = ag_test['title'].fillna('') + ' ' + ag_test['description'].fillna('')
    elif len(ag_test.columns) >= 3:
        col_names = ag_test.columns.tolist()
        ag_test['text'] = ag_test[col_names[1]].fillna('') + ' ' + ag_test[col_names[2]].fillna('')
    else:
        text_col = ag_test.columns[1] if len(ag_test.columns) > 1 else ag_test.columns[0]
        ag_test['text'] = ag_test[text_col]
    
    ag_test['source'] = 'AG_Test'
    datasets.append(ag_test)
    print(f"✓ Loaded AG_Test.csv: {ag_test.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load AG_Test.csv: {e}")

# Combine all datasets
df = pd.concat(datasets, ignore_index=True)

print("-"*60)
print(f"Total combined rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

# Check label distribution
print("\nLabel Distribution:")
print(df['label'].value_counts())


LOADING AND COMBINING DATASETS
✓ Loaded fake.csv: 23,481 rows
✓ Loaded True.csv: 21,417 rows
✓ Loaded AG_Train.csv: 120,000 rows
✓ Loaded AG_Test.csv: 7,600 rows
------------------------------------------------------------
Total combined rows: 172,498
Columns: ['title', 'text', 'subject', 'date', 'label', 'source', 'Class Index', 'Title', 'Description', 'category']

Label Distribution:
label
0    87281
1    85217
Name: count, dtype: int64


In [4]:
# ============================================
# STEP 3: SOURCE VERIFICATION FUNCTION
# ============================================

def source_verification(url):
    """
    Verify the credibility of a news source based on URL
    
    Parameters:
    -----------
    url : str
        The URL of the news article
        
    Returns:
    --------
    dict : Source verification results
    """
    # Extract domain from URL
    try:
        domain = urlparse(url).netloc
        domain = domain.replace("www.", "")
    except:
        domain = "unknown"
    
    # Check if domain is in sources database
    if domain in sources["source"].values:
        score = sources.loc[
            sources["source"] == domain,
            "score"
        ].values[0]
        source_status = "Known Source"
    else:
        score = 50  # Default score for unknown sources
        source_status = "Unknown Source"
    
    # Determine reputation
    if score >= 90:
        reputation = "High"
        risk = "Safe"
    elif score >= 70:
        reputation = "Medium"
        risk = "Moderate"
    else:
        reputation = "Low"
        risk = "Suspicious"
    
    # Check HTTPS
    https_status = "Secure" if url.startswith("https") else "Not Secure"
    
    # Check for spam keywords in URL
    spam_keywords = [
        "breaking", "viral", "secret", "shocking", 
        "click", "free", "miracle", "money", "revealed",
        "exclusive", "urgent", "alert", "breaking", "news"
    ]
    
    spam_indicators = []
    for word in spam_keywords:
        if word in url.lower():
            spam_indicators.append(word)
    
    # Domain length check (suspicious domains are often long)
    domain_length = len(domain)
    if domain_length > 25:
        domain_type = "Suspicious Domain"
    else:
        domain_type = "Normal Domain"
    
    # Compile results
    results = {
        'url': url,
        'domain': domain,
        'domain_type': domain_type,
        'source_status': source_status,
        'credibility_score': score,
        'reputation': reputation,
        'risk_level': risk,
        'https_status': https_status,
        'spam_indicators': spam_indicators,
        'spam_count': len(spam_indicators)
    }
    
    return results

In [5]:
# ============================================
# STEP 4: TEST SOURCE VERIFICATION
# ============================================

print("\n" + "="*60)
print("TESTING SOURCE VERIFICATION")
print("="*60)

test_urls = [
    "https://www.bbc.com/news/world-123",
    "https://viral-secret-news.xyz",
    "https://www.techworldtoday.com",
    "https://www.reuters.com/world",
    "https://www.dawn.com/pakistan",
    "https://www.arynews.tv/business",
    "https://miracle-money-revealed.com"
]

for url in test_urls:
    print(f"\nURL: {url}")
    result = source_verification(url)
    print(f"Domain: {result['domain']}")
    print(f"Source Status: {result['source_status']}")
    print(f"Credibility Score: {result['credibility_score']}")
    print(f"Reputation: {result['reputation']}")
    print(f"Risk Level: {result['risk_level']}")
    print(f"HTTPS: {result['https_status']}")
    print(f"Domain Type: {result['domain_type']}")
    if result['spam_indicators']:
        print(f"⚠️ Spam Indicators: {result['spam_indicators']}")
    else:
        print("✅ No Spam Indicators")
    print("-"*40)


TESTING SOURCE VERIFICATION

URL: https://www.bbc.com/news/world-123
Domain: bbc.com
Source Status: Known Source
Credibility Score: 95
Reputation: High
Risk Level: Safe
HTTPS: Secure
Domain Type: Normal Domain
⚠️ Spam Indicators: ['news']
----------------------------------------

URL: https://viral-secret-news.xyz
Domain: viral-secret-news.xyz
Source Status: Unknown Source
Credibility Score: 50
Reputation: Low
Risk Level: Suspicious
HTTPS: Secure
Domain Type: Normal Domain
⚠️ Spam Indicators: ['viral', 'secret', 'news']
----------------------------------------

URL: https://www.techworldtoday.com
Domain: techworldtoday.com
Source Status: Unknown Source
Credibility Score: 50
Reputation: Low
Risk Level: Suspicious
HTTPS: Secure
Domain Type: Normal Domain
✅ No Spam Indicators
----------------------------------------

URL: https://www.reuters.com/world
Domain: reuters.com
Source Status: Known Source
Credibility Score: 98
Reputation: High
Risk Level: Safe
HTTPS: Secure
Domain Type: Normal 

In [6]:
# ============================================
# STEP 5: ANALYZE SOURCES IN NEWS DATASET
# ============================================

print("\n" + "="*60)
print("ANALYZING SOURCES IN NEWS DATASET")
print("="*60)

# Try to extract domains from text (if URLs are mentioned)
def extract_urls(text):
    """Extract URLs from text"""
    if pd.isna(text):
        return []
    url_pattern = r'https?://[^\s]+'
    return re.findall(url_pattern, str(text))

# Sample a few articles to check for URLs
sample_articles = df['text'].head(100)
url_found = 0
for article in sample_articles:
    urls = extract_urls(article)
    if urls:
        url_found += 1

print(f"Articles with URLs in sample (100 articles): {url_found}")
print("\nNote: This dataset doesn't contain many URLs, so source verification works best with direct URL input")

# Show source distribution
print("\nSource Distribution in Training Data:")
print(df['source'].value_counts())


ANALYZING SOURCES IN NEWS DATASET
Articles with URLs in sample (100 articles): 23

Note: This dataset doesn't contain many URLs, so source verification works best with direct URL input

Source Distribution in Training Data:
source
AG_Train    120000
fake         23481
true         21417
AG_Test       7600
Name: count, dtype: int64


In [7]:
# ============================================
# STEP 6: ENHANCED SOURCE VERIFICATION WITH TEXT ANALYSIS
# ============================================

def enhanced_source_verification(text):
    """
    Enhanced verification that analyzes both text and URLs
    
    Parameters:
    -----------
    text : str
        The news article text
        
    Returns:
    --------
    dict : Enhanced verification results
    """
    results = {
        'mentioned_sources': [],
        'source_scores': [],
        'overall_score': 50,
        'reputation': 'Unknown',
        'risk_level': 'Unknown'
    }
    
    # Extract URLs from text
    urls = extract_urls(text)
    
    # Check each URL
    for url in urls:
        if 'http' in url:
            verification = source_verification(url)
            results['mentioned_sources'].append(verification['domain'])
            results['source_scores'].append(verification['credibility_score'])
    
    # Also check for source mentions (without URLs)
    trusted_sources = ['reuters', 'bbc', 'ap', 'cnn', 'aljazeera', 'dawn', 
                       'nytimes', 'washingtonpost', 'guardian', 'bloomberg']
    
    text_lower = str(text).lower()
    for source in trusted_sources:
        if source in text_lower:
            if source not in results['mentioned_sources']:
                results['mentioned_sources'].append(source)
                # Assign default scores for text mentions
                if source in ['reuters', 'ap']:
                    results['source_scores'].append(95)
                elif source in ['bbc', 'nytimes']:
                    results['source_scores'].append(93)
                elif source in ['dawn', 'aljazeera']:
                    results['source_scores'].append(90)
                else:
                    results['source_scores'].append(85)
    
    # Calculate overall score
    if results['source_scores']:
        results['overall_score'] = sum(results['source_scores']) / len(results['source_scores'])
    else:
        results['overall_score'] = 50  # Default if no sources found
    
    # Determine reputation
    if results['overall_score'] >= 90:
        results['reputation'] = 'High'
        results['risk_level'] = 'Safe'
    elif results['overall_score'] >= 70:
        results['reputation'] = 'Medium'
        results['risk_level'] = 'Moderate'
    else:
        results['reputation'] = 'Low'
        results['risk_level'] = 'Suspicious'
    
    return results

In [8]:
# ============================================
# STEP 7: TEST ENHANCED SOURCE VERIFICATION
# ============================================

print("\n" + "="*60)
print("TESTING ENHANCED SOURCE VERIFICATION")
print("="*60)

test_articles = [
    "According to Reuters, the global economy is showing signs of recovery.",
    "Breaking: Secret government documents revealed by an anonymous source.",
    "The BBC reported today that scientists have made a breakthrough discovery.",
    "A viral video shows shocking footage of a UFO sighting.",
    "As reported by Dawn.com and Al Jazeera, the peace talks have concluded."
]

for article in test_articles:
    print(f"\nArticle: {article}")
    result = enhanced_source_verification(article)
    print(f"Mentioned Sources: {result['mentioned_sources']}")
    print(f"Source Scores: {result['source_scores']}")
    print(f"Overall Score: {result['overall_score']:.2f}")
    print(f"Reputation: {result['reputation']}")
    print(f"Risk Level: {result['risk_level']}")
    print("-"*40)


TESTING ENHANCED SOURCE VERIFICATION

Article: According to Reuters, the global economy is showing signs of recovery.
Mentioned Sources: ['reuters']
Source Scores: [95]
Overall Score: 95.00
Reputation: High
Risk Level: Safe
----------------------------------------

Article: Breaking: Secret government documents revealed by an anonymous source.
Mentioned Sources: []
Source Scores: []
Overall Score: 50.00
Reputation: Low
Risk Level: Suspicious
----------------------------------------

Article: The BBC reported today that scientists have made a breakthrough discovery.
Mentioned Sources: ['bbc']
Source Scores: [93]
Overall Score: 93.00
Reputation: High
Risk Level: Safe
----------------------------------------

Article: A viral video shows shocking footage of a UFO sighting.
Mentioned Sources: []
Source Scores: []
Overall Score: 50.00
Reputation: Low
Risk Level: Suspicious
----------------------------------------

Article: As reported by Dawn.com and Al Jazeera, the peace talks have conclu

In [9]:
# ============================================
# STEP 8: SAVE SOURCES DATABASE
# ============================================

print("\n" + "="*60)
print("SAVING SOURCES DATABASE")
print("="*60)

# Save the sources database
sources.to_csv("sources.csv", index=False)
print("✓ Saved sources database to: sources.csv")

# Save the enhanced verification results for the dataset
print("\nApplying enhanced verification to dataset...")
df['source_mentions'] = df['text'].apply(
    lambda x: enhanced_source_verification(x)['mentioned_sources'] if pd.notna(x) else []
)
df['source_score'] = df['text'].apply(
    lambda x: enhanced_source_verification(x)['overall_score'] if pd.notna(x) else 50
)

print("✓ Added source verification columns to dataset")

# Show summary
print("\nSource Score Distribution:")
print(df['source_score'].describe())

# Save the dataset with source verification
df.to_csv("news_with_source_verification.csv", index=False)
print("✓ Saved dataset with source verification to: news_with_source_verification.csv")


SAVING SOURCES DATABASE
✓ Saved sources database to: sources.csv

Applying enhanced verification to dataset...
✓ Added source verification columns to dataset

Source Score Distribution:
count    172498.000000
mean         71.441551
std          22.077267
min          50.000000
25%          50.000000
50%          50.000000
75%          95.000000
max          96.500000
Name: source_score, dtype: float64
✓ Saved dataset with source verification to: news_with_source_verification.csv


In [10]:
# ============================================
# STEP 9: QUICK VERIFICATION FUNCTION
# ============================================

def verify_news_source(url):
    """
    Quick function to verify a news source
    
    Parameters:
    -----------
    url : str
        The URL to verify
        
    Returns:
    --------
    str : Formatted verification result
    """
    result = source_verification(url)
    
    output = f"""
╔══════════════════════════════════════════════════════════════╗
║                    SOURCE VERIFICATION                      ║
╠══════════════════════════════════════════════════════════════╣
║ URL:           {result['url']}
║ Domain:        {result['domain']}
║ Source Status: {result['source_status']}
║ Score:         {result['credibility_score']}/100
║ Reputation:    {result['reputation']}
║ Risk Level:    {result['risk_level']}
║ HTTPS:         {result['https_status']}
║ Domain Type:   {result['domain_type']}
║ Spam Count:    {result['spam_count']}
╚══════════════════════════════════════════════════════════════╝
    """
    return output

# Test the quick function
print("\n" + "="*60)
print("QUICK VERIFICATION FUNCTION")
print("="*60)

print(verify_news_source("https://www.bbc.com/news/world-123"))
print(verify_news_source("https://viral-secret-news.xyz"))


QUICK VERIFICATION FUNCTION

╔══════════════════════════════════════════════════════════════╗
║                    SOURCE VERIFICATION                      ║
╠══════════════════════════════════════════════════════════════╣
║ URL:           https://www.bbc.com/news/world-123
║ Domain:        bbc.com
║ Source Status: Known Source
║ Score:         95/100
║ Reputation:    High
║ Risk Level:    Safe
║ HTTPS:         Secure
║ Domain Type:   Normal Domain
║ Spam Count:    1
╚══════════════════════════════════════════════════════════════╝
    

╔══════════════════════════════════════════════════════════════╗
║                    SOURCE VERIFICATION                      ║
╠══════════════════════════════════════════════════════════════╣
║ URL:           https://viral-secret-news.xyz
║ Domain:        viral-secret-news.xyz
║ Source Status: Unknown Source
║ Score:         50/100
║ Reputation:    Low
║ Risk Level:    Suspicious
║ HTTPS:         Secure
║ Domain Type:   Normal Domain
║ Spam Count:   

In [11]:
# ============================================
# STEP 10: FINAL SUMMARY
# ============================================

print("\n" + "="*60)
print("FINAL SUMMARY - MODULE 4 COMPLETE")
print("="*60)

print(f"""
Module 4: Source Verification & Credibility Checker

Sources Database:
- Total Sources: {len(sources)}
- High Credibility (90+): {len(sources[sources['score'] >= 90])}
- Medium Credibility (70-89): {len(sources[(sources['score'] >= 70) & (sources['score'] < 90)])}
- Low Credibility (<70): {len(sources[sources['score'] < 70])}

Dataset Analysis:
- Total Articles: {len(df):,}
- Sources Verified: Verified using source_verification()
- Enhanced Verification: Applied to all articles

Functions Created:
✓ source_verification() - Verifies individual URLs
✓ enhanced_source_verification() - Analyzes text for sources
✓ verify_news_source() - Quick formatted verification

Files Saved:
✓ sources.csv - Trusted sources database
✓ news_with_source_verification.csv - Dataset with source verification

Key Features:
• Known source verification
• Spam keyword detection
• HTTPS security check
• Domain length analysis
• Source mention detection
• Credibility scoring
• Risk assessment
""")

print("="*60)
print("MODULE 4 COMPLETE!")
print("="*60)


FINAL SUMMARY - MODULE 4 COMPLETE

Module 4: Source Verification & Credibility Checker

Sources Database:
- Total Sources: 33
- High Credibility (90+): 14
- Medium Credibility (70-89): 19
- Low Credibility (<70): 0

Dataset Analysis:
- Total Articles: 172,498
- Sources Verified: Verified using source_verification()
- Enhanced Verification: Applied to all articles

Functions Created:
✓ source_verification() - Verifies individual URLs
✓ enhanced_source_verification() - Analyzes text for sources
✓ verify_news_source() - Quick formatted verification

Files Saved:
✓ sources.csv - Trusted sources database
✓ news_with_source_verification.csv - Dataset with source verification

Key Features:
• Known source verification
• Spam keyword detection
• HTTPS security check
• Domain length analysis
• Source mention detection
• Credibility scoring
• Risk assessment

MODULE 4 COMPLETE!
